# Day 5, Sales Brouchers

Creeate a product that builds a Brouchure for a company to be used for prospestive cliente, investors and potential recruiters.

We will be provided a company name and their primary website.


In [1]:
import os
import json
import sys
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

# Add the parent folder to Python's import path
sys.path.append("../..")

from week1.scraper import (
    fetch_website_links,
    fetch_website_links,
    fetch_website_contents,
)

In [2]:
load_dotenv(override=True)


api_key = os.getenv("OPENAI_API_KEY")

if api_key and api_key.startswith("sk-proj-") and len(api_key) > 10:
    print("Api key looks so far!")
else:
    print("There's a problem with your API_key, please visit .env")

MODEL = "gpt-5-nano"
openai = OpenAI()

Api key looks so far!


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

# First have gpt-5-nano figure out which links are relevant

Use call to gpt-5 nano to read the links on webpage, and respond in structured JSON

it should decide which are relevant and replace relative links such as './about' with "https://coompany.com/about"

this is a better appreacoh because llms because it reqquires nuanced understanding. Imagine doing this without "llms by parsing and analyzing the webpage, it would be very hard!"


In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
# We are creting a function for getting links and user prompt


def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.
        
Links (some might be relative links):
    
"""

    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [7]:
def select_relevant_links(url):
    print(f"selecting relevant links for {url} -by using {MODEL}.")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)},
        ],
        response_format={"type": "json_object"},
    )

    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"-Found {len(links['links'])} relevant links")
    return links


# def select_relevant_links(url):
#     response = openai.chat.completions.create(
#         model=MODEL,
#         messages=[
#             {"role": "system", "content": link_system_prompt},
#             {"role": "user", "content": get_links_user_prompt(url)},
#         ],
#         response_format={"type": "json_object"},
#     )

#     result = response.choices[0].message.content
#     links = json.loads(result)

#     return links

In [8]:
select_relevant_links("https://edwarddonner.com")

selecting relevant links for https://edwarddonner.com -by using gpt-5-nano.
-Found 5 relevant links


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [9]:
select_relevant_links("https://huggingface.co")

selecting relevant links for https://huggingface.co -by using gpt-5-nano.
-Found 10 relevant links


{'links': [{'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'endpoints product', 'url': 'https://endpoints.huggingface.co'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co/'}]}

# 2nd STEP BROCHURE

- SECOND STEP IS CONVERTING THE FETCHED RELEVANT LINKS IN TO BROCHURE FOROM.


In [10]:
# from llm_engineering.week2.scraper import fetch_website_contents


def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links["links"]:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [11]:
print(fetch_page_and_all_relevant_links("https://edwarddonner.com"))

selecting relevant links for https://edwarddonner.com -by using gpt-5-nano.
-Found 12 relevant links
## Landing Page:

Home - Edward Donner

Skip to content
Avatar
Curriculum
Proficiency
C4
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of AI startup
Nebula.io
. I was previously founder and CEO of AI startup untapt,
acquired in 2021
, and a Managing Director at JPMorgan.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 900,000 enrollment

In [12]:
# brochure_system_prompt = """"
#     You are an assistant that analyzes the contents of several relevant pages from a company's website  and creates a short brochure about the company for the prospective customers, investors and recruits.
#     Respinse in markdown without code blocks.
#     Include details of company's culture, customers and career/jobs if you have the information.
# """


brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website translate into spanish
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

 

In [13]:
def get_brochure_user_prompt(companu_name, url):
    user_prompt= f""""
        You are looking at a company called {companu_name}  Here are the contents of its landing pages;
        use this information to build a short brochure of the company in markdown without code blocks. \n\n
    """
    
    user_prompt+= fetch_page_and_all_relevant_links(url)
    user_prompt= user_prompt[:5_0000]#tuncate if webpage has lot of content
    return user_prompt

In [14]:
get_brochure_user_prompt("Edward's","https://edwarddonner.com")

selecting relevant links for https://edwarddonner.com -by using gpt-5-nano.
-Found 5 relevant links


'"\n        You are looking at a company called Edward\'s  Here are the contents of its landing pages;\n        use this information to build a short brochure of the company in markdown without code blocks. \n\n\n    ## Landing Page:\n\nHome - Edward Donner\n\nSkip to content\nAvatar\nCurriculum\nProficiency\nC4\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of AI startup\nNebula.io\n. I was previously founder and CEO of AI startup untapt,\nacquired in 2021\n, and a Managing Director at JPMorgan.\nI will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and 

# Calling api again based on last call

In [15]:
def create_brochure(company_name, url):
    # print("Generating the brochure")
    response =openai.chat.completions.create(
        model="gpt-4.1-mini" , 
        messages= [{"role": "system" , "content" : brochure_system_prompt}, 
                   {"role": "user" , "content":  get_brochure_user_prompt(company_name,url)}],        
    )
    # print("-Generating the brochure. . .")
    result = response.choices[0].message.content
    print("-Generating the brochure. . .")
    
    display(Markdown(result))

In [16]:
# create_brochure("Edward's Company" , "https://huggingface.co")
create_brochure("Edaward" , "https://edwarddonner.com")


selecting relevant links for https://edwarddonner.com -by using gpt-5-nano.
-Found 7 relevant links
-Generating the brochure. . .


# ¡Bienvenidos a Edaward – Donde la Inteligencia Artificial se vuelve humana… y algo divertida!

---

## ¿Quiénes somos?  

Hola, soy Ed Donner, amante del código, experimentador insaciable de LLMs (modelos de lenguaje enormes, para los que no hablan “AI-nerd”) y cofundador/CTO de Nebula.io. Antes fui CEO de “untapt” (adquirida en 2021) y, para que no piensen que solo juego con robots, también fui Director en JPMorgan.  

Nebula.io es nuestra joya: un startup de AI que revoluciona cómo los reclutadores encuentran talento. Usamos IA Generativa e innovadores modelos patentados que emparejan personas con sus trabajos ideales sin necesidad de buscar palabras clave. ¿Magia? Nah, solo ciencia bien aplicada para que descubrir tu “Ikigai” (tu propósito) no suene a divagación zen.

---

## Nuestra misión:  

En Nebula.io queremos cambiar el mundo laboral… o al menos tu experiencia en el trabajo. Sabías que 77% de las personas no se sienten inspiradas ni comprometidas con su empleo? Nosotros decimos *no más*. Nuestro objetivo es ayudar a que la gente encuentre el trabajo donde pueda brillar, crecer y *amar* lo que hace.  

---

## Cultura Edaward: #CódigoConRitmo

- ¡Nos tomamos el código en serio, pero no tanto a nosotros mismos! Ed mismo produce música electrónica amateur (muy amateur, nos insiste).  
- Nuestra atmósfera es una mezcla de geeks charlatanes – Ed no puede dejar de hablar horas sobre LLMs, pero ahora lo hace hasta en Udemy, donde sus cursos acumulan 900,000 alumnos en 194 países (y contando).  
- Somos curiosos, amantes del aprendizaje continuo (y sí, también pierden la cabeza en Hacker News con nosotros).  
- Nos gustan los retos: “Outsmart” es nuestro campo de batalla, un torneo donde enfrentamos LLMs para ver quién es más astuto, diplomático y ligeramente tramposo.

---

## ¿Eres un futuro Edawarder? ¡Únete!

¿Quieres convertirte en un ingeniero de IA de primera? Ed tiene una completa currícula con 6 cursos que te guían desde “vibe coder” hasta “agentic engineer”, cubriendo desde creación de agentes hasta despliegue en producción. Todos incluyen cupones y la promesa de mantenerte despierto con explicación a prueba de dormilones.  

Completa la currícula, entra a nuestro directorio de Ingenieros AI Proficientes, y tómate la merienda en la cima del futuro tecnológico (podemos prestar el café).  

---

## ¿Clientes felices?  

Desde reclutadores buscando talento hasta líderes de negocio que quieren aplicar IA con impacto real, Nebula.io está marcando diferencia al acelerar y mejorar la contratación con AI. ¿Palabras clave? Olvídalas. Nuestro modelo patentado hace matches a otro nivel… lo justo para que tú solo tengas que elegir qué café tomar mañana.

---

## Contacto y redes sociales

- **Correo:** ed [arroba] edwarddonner [punto] com  
- **Website:** [www.edwarddonner.com](http://www.edwarddonner.com)  
- **LinkedIn, Twitter, Facebook:** Búscanos como Edward Donner y únete a la fiesta de ideas y memes AI.  

---

## Bonus: la vida de Ed fuera de la pantalla

- Charlista frecuente para O’Reilly y Pearson, con charlas que acumulan decenas de miles de vistas.  
- DJ wannabe y eterno aprendiz, porque el código baila mejor con buena música.  

---

# Edaward:  
**Donde la IA se encuentra con el talento humano – sin palabras claves, con mucha pasión y un toque de ritmo.**

¡Descubre tu potencial, aprende sin aburrirte y forma parte de la revolución que está cambiando el futuro del trabajo!

---

*PD: Si escuchas a Ed hablar demasiado de IA, ahora sabes por qué: ¡está preparando su próximo curso Udemy (o su próximo hit electrónico)!*

---

¡Contáctanos, inscríbete en los cursos y únete a Nebula.io para que tu carrera tenga más inteligencia… artificial y natural!

# For chatgpt type, type writer animation
- we gonna use Stream

In [ ]:
def stream_brochure(company_name, url):
    
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)},
        ],
        stream=True,
    )

    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        update_display(Markdown(response), display_id=display_handle.display_id)
        

    

In [18]:
stream_brochure("Edward" , "https://edwarddonner.com")

selecting relevant links for https://edwarddonner.com -by using gpt-5-nano.
-Found 8 relevant links


# ¡Bienvenido a Edward Donner & Nebula.io! 🚀✨

## ¿Quién es Ed?  
Ed Donner es un apasionado de la inteligencia artificial, loco por escribir código y fan (muy amateur) de la producción de música electrónica. Más tech que DJ, más geek que trendsetter, Ed es el co-fundador y CTO de **Nebula.io**, un startup disruptivo en inteligencia artificial que está revolucionando la forma en que las empresas encuentran talento.

Antes de Nebula, Ed fue el fundador y CEO de untapt, otra startup de AI que fue adquirida en 2021. También tuvo una carrera de alto vuelo como Managing Director en JPMorgan. En resumen: Ed sabe lo que hace, y no tiene miedo de hablar de LLMs (modelos de lenguaje grande) hasta aburrir a todo el mundo... ¡tanto que tuvo que crear cursos online para callarse un rato! 

---

## ¿Qué hace Nebula.io?  
Nebula.io es la navaja suiza para reclutadores. Aquí no usamos palabras clave, usamos **modelos patentados de IA generativa** para:

- Encontrar a los mejores candidatos con precisión láser.
- Automatizar todo el proceso de reclutamiento (menos café para ti, más tiempo para dormir).
- Integrarnos sin problema con tus sistemas actuales, para que nada cambie y todo mejore.

**¿El resultado?** Reclutas 3 veces más rápido, reduces un 60% el trabajo manual y contratas con más seguridad. Todo esto mientras ayudas a las personas a encontrar ese trabajo que AMARÁN, que conecta con su "Ikigai" (la razón para levantarse cada día). 🌟

---

## Cultura y Filosofía de Edward & Nebula  
- **Pasión desbordante por la IA:** Aquí, hablamos de agentes autónomos, aprendizaje automático y futuros posibles como si fueran temas de barra de bar.  
- **Apostamos por el talento:** Creemos en descubrir el potencial oculto y dar oportunidades reales.  
- **Diversión con propósito:** Sí, Ed puede aburrirte con un sermón sobre LLMs, pero también monta eventos con O’Reilly y Pearson donde el nerd interior se vuelve estrella... o al menos lo intenta.  
- **Aprender y enseñar:** Con más de 900,000 alumnos en 194 países, los cursos de Ed son una referencia para futuros AI engineers.

---

## ¿Quieres aprender de Ed?  
Ed no solo dirige una startup innovadora, también es un educador de alma. Ofrece 6 cursos de IA que cubren desde construir agentes hasta desplegar modelos a escala, todos con cupones para hacerlos más accesibles. Completa el currículo y únete al selecto directorio de **AI Engineers proficientes** — el lugar donde empleadores y clientes te buscarán. 🎓

---

## Para futuros reclutadores, clientes y colaboradores:  

**¿Buscas un socio tecnológico?** Nebula.io pone el poder de la IA al servicio de tu equipo de reclutamiento, mejorando resultados sin cambiar tu flujo de trabajo.  

**¿Eres un profesional de IA buscando crecer?** Únete a la comunidad, accede al currículo, aprende de un experto que también es un nerd vocacional y encuentra tu camino hasta convertirte en un AI engineer top.  

**¿Quieres invertir o trabajar en un lugar donde la IA se respira y se vive?** Aquí no solo construimos software, construimos futuros.

---

## Datos Curiosos para presumir:  
- Nuestra tecnología ha cruzado las fronteras de lo posible: matching sin palabras claves, 3 veces más rápido que lo convencional.  
- Ed tiene un avatar digital listo para chatear contigo (porque sí, aquí hasta los fundadores están en modo digital).  
- Más de 900,000 personas inscriptas en cursos creados con cariño desde el sótano (o la habitación) de Ed.  
- Eventos en vivo con 180,000 vistas para hablar de agentes autónomos — geek fame nivel proximidad a Twitch.  

---

## Contacto  
Ed y Nebula.io están a un click de distancia:  
📧 ed [at] edwarddonner [dot] com  
🌐 www.edwarddonner.com  
LinkedIn | Twitter | Facebook (cuando funciona)

---

**Nebula.io & Edward Donner: donde la búsqueda del talento se encuentra con la inteligencia artificial… y un toque de humor nerd.**

¡Porque encontrar el candidato ideal y el trabajo perfecto debería ser tan divertido como una batalla de LLMs diplomáticos y astutos!

---

*¿Listo para jugar en las grandes ligas del talento con IA?*  
*Haz clic, conecta y descubre esta revolución.*